# Coordinator and Subagents (Managed Agents API)

> **CCA-F Domain 1 - Agentic (27%).** The exam tests **isolated-context orchestration**: a **coordinator** spawns each specialist in its **own session thread** with its **own fresh context**, and only the specialist's result flows back. The trap is the **daisy-chain anti-pattern** - piping every raw specialist transcript through one ever-growing coordinator context until it blows the window and the reasoning rots.

This notebook builds a **Vampire Survivors build council**: a coordinator that delegates to a **weapon-evolution specialist** and a **synergy specialist**. We show the `multiagent` coordinator shape and **one delegated turn**, then stop. This is the shape lesson, not a full app.

> **These calls are live, billable, and beta-gated.** Read them to learn the shape. Do not run them without Managed Agents beta access.

### 1. Install dependencies

The `anthropic` SDK carries the Managed Agents surface under `client.beta.agents`, `.environments`, and `.sessions`. `python-dotenv` reads the API key from `.env`.

In [ ]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

### 2. Load environment variables and create the client

`load_dotenv()` reads the API key from `.env`. The Managed Agents surface is **beta-gated**, so we set `BETA` once and pass `betas=[BETA]` on every call. `MODEL` stays on **Haiku 4.5** (the repo default); a coordinator plus two specialists at production quality does not need Sonnet.

In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
MODEL = "claude-haiku-4-5"            # unversioned alias -> current Haiku 4.5 snapshot
BETA = "managed-agents-2026-04-01"    # Managed Agents beta header, passed on every call

### 3. Build the specialist roster

Each specialist is a **plain agent** - no `multiagent` config (the depth limit is 1, so a specialist cannot itself be a coordinator). Give each a **narrow `system` prompt** so its context stays focused. When the coordinator spawns one, that specialist runs in its **own session thread with fresh context**; it never sees the other specialist's transcript. That isolation is the whole point of the pattern.

In [ ]:
# Two narrow specialists. Each is a normal agent (no multiagent config).
weapon_specialist = client.beta.agents.create(
    model={"id": MODEL},
    name="weapon-evolution-specialist",
    system=(
        "You are a Vampire Survivors weapon-evolution specialist. Given a run's "
        "current weapons, name which evolutions are reachable and the exact passive "
        "item each evolution requires. Answer only about weapon evolutions."
    ),
    betas=[BETA],
)

synergy_specialist = client.beta.agents.create(
    model={"id": MODEL},
    name="synergy-specialist",
    system=(
        "You are a Vampire Survivors synergy specialist. Given a build's weapons and "
        "passives, call out stat and passive synergies (area, cooldown, projectiles) "
        "and any wasted overlap. Answer only about synergy, not evolutions."
    ),
    betas=[BETA],
)

print("weapon specialist:", weapon_specialist.id)
print("synergy specialist:", synergy_specialist.id)

### 4. Wire the coordinator with the `multiagent` config

The `multiagent` param turns an agent into a **coordinator**. Its shape is verified against the SDK (`BetaManagedAgentsMultiagentParams`): two required keys.

| Key | Value | Notes |
|---|---|---|
| `type` | `"coordinator"` | The only allowed value. |
| `agents` | roster list, 1-20 entries | Each entry is an **agent-ID string** (simplest), a versioned `{"type":"agent","id":...,"version":...}` ref, or `{"type":"self"}` for recursion. Referenced agents must exist, must not be archived, and must not themselves set `multiagent` (**depth limit 1**). |

We use the **string form** - the simplest roster entry - and stay minimal: `type` + `agents` only. The coordinator's `system` prompt is the routing brief; it decides which specialist gets which sub-question, spawns each in an isolated thread, and composes the results.

In [ ]:
# The multiagent config is the coordinator shape. Roster entries are the
# specialist agent IDs (string form). Depth limit is 1, so neither specialist
# may itself carry a multiagent config.
council = client.beta.agents.create(
    model={"id": MODEL},
    name="build-council-coordinator",
    system=(
        "You are the Vampire Survivors build council coordinator. Break the player's "
        "request into an evolution question and a synergy question, delegate each to "
        "the matching specialist, then compose one build recommendation. Delegate; "
        "do not answer the specialist questions yourself."
    ),
    multiagent={
        "type": "coordinator",
        "agents": [weapon_specialist.id, synergy_specialist.id],
    },
    betas=[BETA],
)

print("coordinator:", council.id)

### 5. Run one delegated turn

Same spine as any Managed Agents session: **create an environment**, **create a session** bound to the coordinator, **send** the user turn, then **stream** events until the session goes idle. When the coordinator spawns a specialist you see delegation events; the specialist's isolated thread runs, and only its result returns. **Stop on `session.status_idle`** - that event carries `stop_reason`, not the streamed prose. Stopping on vibes is the exam trap.

In [ ]:
# Environment + session bound to the coordinator agent.
env = client.beta.environments.create(name="build-council-scratch", betas=[BETA])
session = client.beta.sessions.create(
    agent=council.id,             # agent accepts the agent ID string
    environment_id=env.id,
    betas=[BETA],
)

# Send one player turn.
client.beta.sessions.events.send(
    session.id,
    events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": (
                "My run has King Bible and Garlic, plus Empty Tome and Pummarola. "
                "What should I chase next, and does anything synergize or overlap?"
            ),
        }],
    }],
    betas=[BETA],
)

# Stream until the session goes idle. Stop on the status event, not the prose.
for event in client.beta.sessions.events.stream(session.id, betas=[BETA]):
    if event.type == "agent.message":
        print("[coordinator]", event.type)
    elif event.type == "session.status_idle":
        print("idle - stop_reason:", getattr(event, "stop_reason", None))
        break

### 6. Tear down

Always **archive** what you created so no billable session dangles. There is no `agents.delete` - archive is the retirement path for both agents and sessions. Archive the session first, then the coordinator, then both specialists.

In [ ]:
# Teardown: archive the session, then every agent. No agents.delete exists.
client.beta.sessions.archive(session.id, betas=[BETA])
client.beta.agents.archive(council.id, betas=[BETA])
client.beta.agents.archive(weapon_specialist.id, betas=[BETA])
client.beta.agents.archive(synergy_specialist.id, betas=[BETA])
print("Archived session + 3 agents. Nothing left billable.")